# 02 — Gerar agents e aplicar questionário com OpenAI

Este notebook lê a base `.csv` da população de agentes, transforma cada linha em um agente sintético por prompt e aplica um questionário a um artefato de texto.

Configuração padrão:

- Modelo: `gpt-4o-mini`
- Modo OpenAI: ativado
- Saídas:
  - respostas individuais em `.jsonl`
  - respostas individuais em `.csv`
  - agregado por construct
  - agregado por segmento


In [5]:
from pathlib import Path
import os
import sys
import json
import time
import uuid
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 220)

# Ajuste este caminho para a pasta do seu projeto local
PROJECT_DIR = Path(r"C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm")

OUTPUT_DIR = PROJECT_DIR / "data" / "output"
INPUT_DIR = PROJECT_DIR / "data" / "input"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("INPUT_DIR:", INPUT_DIR)


PROJECT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm
OUTPUT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output
INPUT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\input


## 1. Configurar OpenAI

Antes de rodar, garanta que sua chave está configurada.

No Windows PowerShell:

```powershell
setx OPENAI_API_KEY "sua-chave"
```

Depois feche e reabra o Jupyter/VS Code.

Também pode configurar temporariamente dentro do notebook, mas não recomendo salvar chave no arquivo.


In [ ]:
# Instalar, se necessário:
# !pip install openai

from openai import OpenAI

MODEL = "gpt-4o-mini"

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY não encontrada. Configure a variável de ambiente antes de rodar."
    )

client = OpenAI()

print("OpenAI configurada.")
print("Modelo:", MODEL)


OSError: OPENAI_API_KEY não encontrada. Configure a variável de ambiente antes de rodar.

## 2. Ler base CSV dos agentes

Use o CSV criado no notebook anterior.

Se o arquivo tiver outro nome, altere `AGENTS_CSV`.


In [ ]:
# Opção A: caminho explícito
AGENTS_CSV = PROJECT_DIR / "data" / "output" / "agents_population_n1000_seed42.csv"

# Opção B: pegar automaticamente o primeiro CSV de população encontrado
if not AGENTS_CSV.exists():
    candidate_files = sorted(OUTPUT_DIR.glob("agents_population*.csv"))
    if candidate_files:
        AGENTS_CSV = candidate_files[0]

print("AGENTS_CSV:", AGENTS_CSV)
print("Existe?", AGENTS_CSV.exists())

df_agents = pd.read_csv(AGENTS_CSV)

print("Agentes carregados:", len(df_agents))
df_agents.head(3)


## 3. Definir questionário

Este questionário é inicial e pode ser substituído depois por um CSV padronizado.

A recomendação para o MVP é usar escala Likert de 1 a 5.


In [ ]:
questionnaire = [
    {
        "question_id": "Q001",
        "construct": "confiança_institucional",
        "question": "Em uma escala de 1 a 5, quanto você confia na informação apresentada neste texto?",
        "scale_min": 1,
        "scale_max": 5,
        "scale_labels": {
            1: "Não confio nada",
            2: "Confio pouco",
            3: "Confio parcialmente",
            4: "Confio bastante",
            5: "Confio totalmente",
        },
    },
    {
        "question_id": "Q002",
        "construct": "percepção_de_risco",
        "question": "Em uma escala de 1 a 5, quanto este texto aumenta sua sensação de risco ou preocupação?",
        "scale_min": 1,
        "scale_max": 5,
        "scale_labels": {
            1: "Não aumenta nada",
            2: "Aumenta pouco",
            3: "Aumenta moderadamente",
            4: "Aumenta bastante",
            5: "Aumenta muito",
        },
    },
    {
        "question_id": "Q003",
        "construct": "probabilidade_de_compartilhamento",
        "question": "Em uma escala de 1 a 5, qual a chance de você compartilhar ou comentar este conteúdo?",
        "scale_min": 1,
        "scale_max": 5,
        "scale_labels": {
            1: "Muito improvável",
            2: "Pouco provável",
            3: "Talvez",
            4: "Provável",
            5: "Muito provável",
        },
    },
    {
        "question_id": "Q004",
        "construct": "valência_percebida",
        "question": "Em uma escala de 1 a 5, qual é o tom geral percebido neste texto?",
        "scale_min": 1,
        "scale_max": 5,
        "scale_labels": {
            1: "Muito negativo",
            2: "Negativo",
            3: "Neutro ou ambivalente",
            4: "Positivo",
            5: "Muito positivo",
        },
    },
]

df_questionnaire = pd.DataFrame(questionnaire)
QUESTIONNAIRE_CSV = INPUT_DIR / "questionnaire_example.csv"
df_questionnaire.to_csv(QUESTIONNAIRE_CSV, index=False, sep=";", encoding="utf-8-sig")

df_questionnaire


## 4. Definir artefato analisado

Troque o texto abaixo por uma notícia, release, post ou qualquer conteúdo que você queira testar.


In [ ]:
artifact = {
    "artifact_id": "NEWS_001",
    "artifact_type": "news_text",
    "title": "Empresa anuncia investimento em tecnologia e sustentabilidade",
    "content": """
Uma grande empresa brasileira anunciou um novo plano de investimento em tecnologia,
inteligência artificial e sustentabilidade. Segundo a companhia, a iniciativa deve aumentar
a eficiência operacional, reduzir emissões e abrir novas oportunidades de emprego qualificado.

Especialistas avaliam que o plano pode fortalecer a imagem institucional da empresa,
mas organizações da sociedade civil cobram mais transparência sobre metas ambientais,
governança dos dados e impactos sobre trabalhadores em funções operacionais.
""".strip(),
}

ARTIFACT_JSON = INPUT_DIR / "artifact_news_001.json"
ARTIFACT_JSON.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")

artifact


## 5. Funções de prompt dos agents

In [ ]:
def safe_get(row, col, default=""):
    value = row[col] if col in row.index else default
    if pd.isna(value):
        return default
    return str(value)


def build_prompt_user(agent_row: pd.Series) -> str:
    return f"""
Você é um respondente sintético brasileiro em uma simulação de audiência.

Seu perfil demográfico:
- ID da persona: {safe_get(agent_row, "id_persona")}
- Idade: {safe_get(agent_row, "idade")}
- Gênero: {safe_get(agent_row, "gênero")}
- Raça/cor: {safe_get(agent_row, "raça/cor")}
- Escolaridade: {safe_get(agent_row, "escolaridade")}
- Situação ocupacional: {safe_get(agent_row, "situação ocupacional")}
- Renda familiar mensal: {safe_get(agent_row, "renda familiar mensal")}
- Localização: {safe_get(agent_row, "localização")}
- UF: {safe_get(agent_row, "uf")}
- Tipo de território: {safe_get(agent_row, "tipo_de_território")}
- Identificação política: {safe_get(agent_row, "identificação política")}
- Ideologia: {safe_get(agent_row, "ideologia")}

Seu perfil survey-based:
- Geração: {safe_get(agent_row, "geração")}
- Segmento e bolha: {safe_get(agent_row, "segmento_e_bolhas")}
- Fontes de informação: {safe_get(agent_row, "tipo_de_fontes_de_informação")}

Valores, atitudes e percepções:
- Religiosidade: {safe_get(agent_row, "Religiosidade")}
- Família: {safe_get(agent_row, "Família")}
- Honestidade: {safe_get(agent_row, "Honestidade")}
- Meritocracia: {safe_get(agent_row, "Meritocracia")}
- Ideologia: {safe_get(agent_row, "Ideologia")}
- Discriminação: {safe_get(agent_row, "Discriminação")}
- Sexualidade: {safe_get(agent_row, "Sexualidade")}
- Comportamento social: {safe_get(agent_row, "Comportamento_social")}
- Identidade nacional: {safe_get(agent_row, "Identidade_nacional")}
- Insegurança: {safe_get(agent_row, "Insegurança")}
- Confiança: {safe_get(agent_row, "Confiança")}

Descrição da persona:
{safe_get(agent_row, "Persona")}

Responda às perguntas como essa pessoa tenderia a responder, não como um analista neutro.
""".strip()


def build_prompt_system() -> str:
    return """
Você é um simulador de audiência sintética baseado em Agent-Based Modeling e Cognitive Modeling.

Sua tarefa é responder questionários de pesquisa como se fosse uma persona específica.

Regras:
1. Use apenas o perfil da persona e o artefato analisado.
2. Responda em escala Likert quando solicitado.
3. Mantenha coerência com geração, segmento, renda, território, política, valores e fontes de informação.
4. Não responda como especialista ou como modelo de IA.
5. Retorne sempre JSON válido.

Formato obrigatório:
{
  "answer_value": número inteiro dentro da escala,
  "answer_label": "rótulo da escala",
  "rationale": "justificativa curta em português, com no máximo 2 frases",
  "confidence": número entre 0 e 1
}
""".strip()


def build_prompt_reasoning() -> str:
    return """
Thinking mode:
Antes de responder, considere silenciosamente:
1. Como o segmento social da persona tende a interpretar o tema.
2. Como geração, renda, ideologia, religiosidade e território afetam a leitura.
3. Se o texto ativa confiança, risco, rejeição, identificação ou vontade de compartilhar.
4. Escolha uma resposta Likert coerente com esse perfil.

Não exponha raciocínio longo. Retorne apenas o JSON final.
""".strip()


def build_question_prompt(question: dict, artifact: dict) -> str:
    labels = "\n".join([f"{k}: {v}" for k, v in question["scale_labels"].items()])

    return f"""
Artefato analisado:
ID: {artifact["artifact_id"]}
Tipo: {artifact["artifact_type"]}
Título: {artifact["title"]}

Conteúdo:
{artifact["content"]}

Pergunta:
{question["question"]}

Construct:
{question["construct"]}

Escala:
{labels}

Responda apenas com JSON válido.
""".strip()


def build_agent_messages(agent_row: pd.Series, question: dict, artifact: dict) -> list:
    return [
        {"role": "system", "content": build_prompt_system()},
        {"role": "user", "content": build_prompt_user(agent_row)},
        {"role": "user", "content": build_prompt_reasoning()},
        {"role": "user", "content": build_question_prompt(question, artifact)},
    ]


example_messages = build_agent_messages(df_agents.iloc[0], questionnaire[0], artifact)
example_messages


## 6. Função para chamar OpenAI

In [ ]:
def parse_json_response(text: str) -> dict:
    text = text.strip()

    if text.startswith("```"):
        text = text.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end+1])
        raise


def openai_agent_response(agent_row: pd.Series, question: dict, artifact: dict, model: str = MODEL) -> dict:
    messages = build_agent_messages(agent_row, question, artifact)

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.4,
        response_format={"type": "json_object"},
    )

    text = response.choices[0].message.content
    data = parse_json_response(text)

    # Normalização defensiva
    data["answer_value"] = int(data["answer_value"])
    data["confidence"] = float(data.get("confidence", 0.75))

    return data


# Teste rápido com 1 agente e 1 pergunta
test_answer = openai_agent_response(df_agents.iloc[0], questionnaire[0], artifact)
test_answer


## 7. Rodar os agents

Comece com poucos agentes, depois aumente.

Sugestão:

1. `LIMIT_AGENTS = 10`
2. `LIMIT_AGENTS = 100`
3. `LIMIT_AGENTS = None` para rodar todos

`REPEATS = 3` roda cada pergunta três vezes por agente e permite calcular média/desvio padrão. Para economizar e acelerar, use `REPEATS = 1`.


In [ ]:
LIMIT_AGENTS = 10      # use None para rodar todos
REPEATS = 3            # 1, 3 ou 5
SLEEP_SECONDS = 0.2    # pequena pausa entre chamadas

RUN_ID = f"run_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"

RESPONSES_JSONL = OUTPUT_DIR / f"{RUN_ID}_responses.jsonl"
RESPONSES_CSV = OUTPUT_DIR / f"{RUN_ID}_responses.csv"
AGG_CSV = OUTPUT_DIR / f"{RUN_ID}_aggregated.csv"

if LIMIT_AGENTS is None:
    df_run_agents = df_agents.copy()
else:
    df_run_agents = df_agents.head(LIMIT_AGENTS).copy()

total_calls = len(df_run_agents) * len(questionnaire) * REPEATS

print("RUN_ID:", RUN_ID)
print("Modelo:", MODEL)
print("Agentes:", len(df_run_agents))
print("Perguntas:", len(questionnaire))
print("Repeats:", REPEATS)
print("Total de chamadas OpenAI:", total_calls)
print("JSONL:", RESPONSES_JSONL)
print("CSV:", RESPONSES_CSV)


In [ ]:
records = []
errors = []

with RESPONSES_JSONL.open("w", encoding="utf-8") as f:
    call_n = 0

    for agent_idx, agent_row in df_run_agents.iterrows():
        for question in questionnaire:
            for repeat_id in range(REPEATS):
                call_n += 1

                try:
                    answer = openai_agent_response(
                        agent_row=agent_row,
                        question=question,
                        artifact=artifact,
                        model=MODEL,
                    )

                    record = {
                        "run_id": RUN_ID,
                        "model": MODEL,
                        "artifact_id": artifact["artifact_id"],
                        "artifact_type": artifact["artifact_type"],
                        "artifact_title": artifact["title"],
                        "id_persona": safe_get(agent_row, "id_persona"),
                        "segmento_e_bolhas": safe_get(agent_row, "segmento_e_bolhas"),
                        "geração": safe_get(agent_row, "geração"),
                        "idade": safe_get(agent_row, "idade"),
                        "gênero": safe_get(agent_row, "gênero"),
                        "raça/cor": safe_get(agent_row, "raça/cor"),
                        "renda familiar mensal": safe_get(agent_row, "renda familiar mensal"),
                        "uf": safe_get(agent_row, "uf"),
                        "tipo_de_território": safe_get(agent_row, "tipo_de_território"),
                        "identificação política": safe_get(agent_row, "identificação política"),
                        "ideologia": safe_get(agent_row, "ideologia"),
                        "question_id": question["question_id"],
                        "construct": question["construct"],
                        "question": question["question"],
                        "repeat_id": repeat_id,
                        "answer_value": answer.get("answer_value"),
                        "answer_label": answer.get("answer_label"),
                        "rationale": answer.get("rationale"),
                        "confidence": answer.get("confidence"),
                        "raw_answer": json.dumps(answer, ensure_ascii=False),
                    }

                    records.append(record)
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")

                except Exception as e:
                    error_record = {
                        "call_n": call_n,
                        "id_persona": safe_get(agent_row, "id_persona"),
                        "question_id": question["question_id"],
                        "repeat_id": repeat_id,
                        "error": repr(e),
                    }
                    errors.append(error_record)
                    print("ERRO:", error_record)

                if call_n % 10 == 0:
                    print(f"Progresso: {call_n}/{total_calls}")

                time.sleep(SLEEP_SECONDS)

df_responses = pd.DataFrame(records)
df_responses.to_csv(RESPONSES_CSV, index=False, sep=";", encoding="utf-8-sig")

print("Finalizado.")
print("Respostas válidas:", len(df_responses))
print("Erros:", len(errors))
print("CSV salvo em:", RESPONSES_CSV)

df_responses.head()


## 8. Agregar resultados

Aqui calculamos média, desvio padrão, número de respostas e confiança média.

O resultado pode ser analisado por:

- total
- segmento
- geração
- UF
- ideologia


In [ ]:
df_responses["answer_value"] = pd.to_numeric(df_responses["answer_value"], errors="coerce")
df_responses["confidence"] = pd.to_numeric(df_responses["confidence"], errors="coerce")

def aggregate_by(group_cols):
    return (
        df_responses
        .groupby(group_cols, as_index=False)
        .agg(
            mean_answer=("answer_value", "mean"),
            std_answer=("answer_value", "std"),
            n=("answer_value", "count"),
            mean_confidence=("confidence", "mean"),
        )
    )


agg_total = aggregate_by(["artifact_id", "construct", "question_id"])
agg_total["group_type"] = "total"
agg_total["group_value"] = "total"

agg_segment = aggregate_by(["artifact_id", "construct", "question_id", "segmento_e_bolhas"])
agg_segment["group_type"] = "segmento_e_bolhas"
agg_segment = agg_segment.rename(columns={"segmento_e_bolhas": "group_value"})

agg_generation = aggregate_by(["artifact_id", "construct", "question_id", "geração"])
agg_generation["group_type"] = "geração"
agg_generation = agg_generation.rename(columns={"geração": "group_value"})

agg_ideology = aggregate_by(["artifact_id", "construct", "question_id", "ideologia"])
agg_ideology["group_type"] = "ideologia"
agg_ideology = agg_ideology.rename(columns={"ideologia": "group_value"})

df_agg = pd.concat(
    [agg_total, agg_segment, agg_generation, agg_ideology],
    ignore_index=True
)

for col in ["mean_answer", "std_answer", "mean_confidence"]:
    df_agg[col] = df_agg[col].round(3)

df_agg.to_csv(AGG_CSV, index=False, sep=";", encoding="utf-8-sig")

print("Agregado salvo em:", AGG_CSV)
df_agg.head(30)


## 9. Tabela executiva por segmento

In [ ]:
pivot_segment = (
    df_agg[df_agg["group_type"] == "segmento_e_bolhas"]
    .pivot_table(
        index="group_value",
        columns="construct",
        values="mean_answer",
        aggfunc="mean",
    )
    .round(2)
)

EXECUTIVE_SUMMARY_CSV = OUTPUT_DIR / f"{RUN_ID}_executive_summary_by_segment.csv"
pivot_segment.to_csv(EXECUTIVE_SUMMARY_CSV, sep=";", encoding="utf-8-sig")

print("Resumo executivo salvo em:", EXECUTIVE_SUMMARY_CSV)
pivot_segment


## 10. Inspecionar justificativas

Essa célula ajuda a auditar se os agentes estão respondendo com coerência.


In [ ]:
df_responses[
    [
        "id_persona",
        "segmento_e_bolhas",
        "geração",
        "construct",
        "answer_value",
        "answer_label",
        "rationale",
        "confidence",
    ]
].sample(min(10, len(df_responses)), random_state=42)


## 11. Próximos passos

Depois que este notebook funcionar, recomendo modularizar em:

```text
src/synthetic_audiences/
├── agents/
│   ├── prompts.py
│   ├── runner.py
│   └── parser.py
├── surveys/
│   ├── questionnaire.py
│   └── aggregation.py
└── artifacts/
    └── loaders.py
```

O notebook deve virar apenas uma interface de execução.
